In [ ]:
import sys,os,shutil,getpass
# the work dir must be in the root of "Source"
work_dir = os.getcwd()
if 'notebooks' in work_dir :
  parent_dir = os.path.dirname(os.getcwd())
sys.path.append(parent_dir)

In [ ]:
import datetime
import numpy as np
import subprocess
import folium
import random
import matplotlib.pyplot as plt
import copernicusmarine as cm

from copernicus_functions_poly import *
from urllib.parse import urlparse
from folium import plugins
from netCDF4 import Dataset,num2date
from shapely.geometry import box, Point, Polygon

# SOURCE input data set
***

### IN-SITU DATA FROM __MOORED TEMPERATURE AND SALINITY__ OBSERVATIONS
SOURCE implementation uses Near Real Time __moored__ temperature and salinity observation from CMEMS Service. 
First of all you have to create a CMEMS account following the instruction available here: https://data.marine.copernicus.eu/register. 
Each CMEMS ocean product has a dedicated section (__DATA ACCESS__) where you can find all the instruction on how download the data. Please read also the __User Manual__ describing the interested product and how it is organized.

Here they are used observation from __cmems_obs-ins_med_phybgcwav_mynrt_na_irr__ product covering the Global seas, considering only data in Mediterranean region. 

To facilitate navigation and automatic download, the In Situ TAC provides a set of `index files` that describe the netCDF collections. 



| Index |  Description |
| :- | :- |
| `index_latest.txt`  |  providing access to a sliding window on the latest 31      days (     the current day included) of observations for real-time applications  | 
| `index_monthly.txt`   | accumulating the best copy of a dataset, organised      by platform and by month, for the last 5 years (plus the current year until the last month, not including the current month) |
| `index_history.txt`   |  providing access to the best quality copy of an observation organised by platform  |



#### 1. Define your time window

In [ ]:
targeted_range = '2016-01-01T00:00:00Z/2016-12-31T00:00:00Z' #set your own!

#### 2. Enter CMEMS username and password
After the registration, you will be able to dowload the data using __Copernicus Marine Toolbox Command Line Interface (CLI)__ .

Please `run the next` to create a configuration file with your Copernicus Marine credentials

In [ ]:
cm.login()

As stated before, SOURCE focuses on the Near Real Time product/dataset covering the Mediterranean sea coming from Global seas product. Following steps were taken from useful python sample codes developed by CMEMS service and describing in the __Product User Manual for *insitu* product__ .

Please `run the next` to load the info defining such product/dataset:

#### 3. Define CMEMS product and dataset

In [ ]:
dataset ='cmems_obs-ins_glo_phybgcwav_mynrt_na_irr'
dataset_time_period='history'

first_dataset_type='OBSERVATION'

dataset_details = {
    'product': 'INSITU_GLO_PHYBGCWAV_DISCRETE_MYNRT_013_030',#name of the In Situ Near Real Time product in the MED area
    'name': 'cmems_obs-ins_med_phybgcwav_mynrt_na_irr',#name of the dataset available in the above In Situ Near Real Time product
    'index_files': ['index_'+dataset_time_period+'.txt'],#file describing the content of the history netCDF file collections available withint he above dataset
    'index_platform': 'index_platform.txt',#files describing the netwotk of platforms contributting with files in the abve collections
}

In [ ]:
input_directory=parent_dir+'/input_NA/OBSERVATION/'
if not os.path.exists(input_directory):
    os.makedirs(input_directory)
output_directory=parent_dir+'/output_NA/'
if not os.path.exists(output_directory):
    os.mkdir(output_directory)

#### 4. Download the index files

In [ ]:
cm.get(dataset_id=dataset,
       force_download = True,
       output_directory = input_directory,
       no_directories = True,
       index_parts = True)

#### 5. Define target area 

As the __cmems_obs-ins_med_phybgcwav_mynrt_na_irr__ product covers all the world ocean, it is possible to set next a bounding box of interest in `the next cell` and run it:

In [ ]:
geospatial_lat_min = 30.0  # enter min latitude of your bounding box
geospatial_lat_max = 46  # enter max latitude of your bounding box
geospatial_lon_min = -6 # enter min longitude of your bounding box
geospatial_lon_max = 37  # enter max longitude of your bounding box
targeted_bbox = [geospatial_lon_min, geospatial_lat_min, geospatial_lon_max, geospatial_lat_max]  # (minx, miny, maxx, maxy)

p1=Point(30, -6)#18.5)
p2=Point(43.2, -.5)
p3=Point(43.2, -0.8)
p4=Point(43.2, -0.8)
p5=Point(46.5, 16)
p6=Point(41.8, 26.8)
p7=Point(40.3, 26.8)
p8=Point(38, 36.5)
p9=Point(30, 36.5)

points=[p1,p2,p3,p4,p5,p6,p7,p8,p9]

# Create a polygon from a list of shapely points
med_poly = Polygon([[p.x, p.y] for p in points])


Let's see it on a map, `run the next cell`

In [ ]:
m = folium.Map(location=[53.0, 0], zoom_start=4)
upper_left = [geospatial_lat_max, geospatial_lon_min]
upper_right = [geospatial_lat_max, geospatial_lon_max]
lower_right = [geospatial_lat_min, geospatial_lon_max]
lower_left = [geospatial_lat_min, geospatial_lon_min]
edges_ = [upper_left, upper_right, lower_right, lower_left]
edges_ = [ [ 30, -6], [ 43.2, -6], [43.2, -0.8], [ 43.2, -0.8], [ 46.5, 16], [ 41.8, 26.8], [40.3, 26.8], [ 38, 36.5],[ 30, 36.5]]
polygon = folium.vector_layers.Polygon(locations=edges_)
m.add_child(polygon)
m.fit_bounds(polygon.get_bounds())
m

In [ ]:
info = getIndexFilesInfo(dataset_details,input_directory,targeted_bbox, med_poly)
info['poligonOverlap'] = info.apply(poligonOverlap,targeted_bbox=targeted_bbox,med_poly=med_poly, axis=1)


#### 6. Obtain the interested data info subsetting by bounding-box

`Run the next cell` to obtain the subset of files with data in such area:

In [ ]:
info['poligonOverlap'] = info.apply(poligonOverlap,targeted_bbox=targeted_bbox,med_poly=med_poly, axis=1)
condition1 = info['poligonOverlap'] == True
subset = info[condition1]
subset.transpose()

#### 7. Obtain the interested data info subsetting by time-range

In [ ]:
info['timeOverlap'] = info.apply(timeOverlap,targeted_range=targeted_range,axis=1)
condition2 = info['timeOverlap'] == True
subset = info[condition2]
subset.transpose()

#### 8. Obtain the interested data info subsetting by data-type

In [ ]:
targeted_data_type = 'MO'
condition3 = info['data_type'] == targeted_data_type
subset = info[condition3]
subset.transpose()

In [ ]:
targeted_file_type = 'TS'
condition4 = info['file_type'] == targeted_file_type
subset = info[condition4]
subset.transpose()

#### 9. Subsetting by several criterias at once

In [ ]:
subset = info[condition1 & condition2 & condition3 & condition4]
subset.transpose()

#### 10. Downloading

After you have created your own subset (see above examples about how-to), we will create the list of the files to download.
`Run the next cells`:

In [ ]:
with open('list_files_to_download.txt','w') as list_txt:
 for i in np.arange(0,np.size(subset.file_name)):
    list_txt.write(dataset_time_period+'/'+str(os.path.join(os.path.split(subset.file_name.iloc[i])[0].split('/')[-1::][0],os.path.split(subset.file_name.iloc[i])[1]))+'\n')

This list has been saved into list_files_to_download.txt, that we can now use to launch the download using --file-list

In [ ]:
cm.get(dataset_id=dataset,
       force_download = True,
       output_directory = input_directory+'/'+targeted_data_type,
       no_directories = True,
       file_list = 'list_files_to_download.txt')

#### 11. Visualize the position of the downloaded data

In [ ]:
numberOfFiles = 2000 #we will check just a sample of files not all

In [ ]:
m = folium.Map(location=[39.3, 0], zoom_start=5)
m.add_child(folium.vector_layers.Polygon(locations=edges_))
for platform, files in subset[:numberOfFiles].groupby(['platform_code', 'data_type']):
    color = "%06x" % random.randint(0, 0xFFFFFF)
    #Last reported position to map as marker
    i = len(files)-1
    m.add_child(folium.Marker([files.iloc[i]['last_latitude_observation'], files.iloc[i]['last_longitude_observation']], popup=files.iloc[i]['platform_code']+' last position' ))
#Zooming closer
m.fit_bounds(edges_, max_zoom=8)
m